# Large Repricing Prediction Benchmark

## Paper Narrative

The main story here is that **a large repricing event can be interpreted as an observable posterior revision of the market's belief state**.

This benchmark answers the core paper question:
- can we identify in advance the states where the market is likely to be revised sharply,
- are such states linked to missing information, external shocks, or internal incoherence,
- and does richer context improve over a simple volatility heuristic.

For the paper, this is the bridge benchmark between raw market traces and the broader story about information arrival, trust, and coherence.

## Everyday Intuition: Why This Might Work

The intuition is simple:
- if the market has not fully absorbed information yet, we can often see it in advance through jumpy trajectories and rising local instability;
- some states are “tense” in the sense that they are much more likely to be revised soon than others;
- the goal is therefore not to predict the final outcome directly, but to recognize when the current belief state is fragile.

## What Data We Will Use

- rolling snapshots from `probabilities`;
- recent `yes_probability` history for each market;
- market metadata from `markets`;
- when available: coherence, manipulation, and external-shock covariates.

## What Metrics To Track

- `average_precision` as the key metric for a rare event;
- `roc_auc`;
- `log_loss`;
- descriptive plots by pressure and shock buckets.

## What Models To Train

- a volatility-only heuristic baseline;
- a market-only tree baseline;
- a richer multimodal baseline;
- when available, a separate external-shock baseline.

## How To Read This Notebook

This is a research tutorial for the repricing-detection task.

Reading order:
1. We define what a large repricing is and why it is a proxy for a belief update.
2. We build the dataset from the 5-minute market trajectory.
3. We fix an evaluation protocol without temporal leakage.
4. We compare a volatility baseline, market-only models, and stronger models.
5. We inspect whether repricing events concentrate in “high-pressure” market states.

## Task

**What we predict:**
- whether a large revision of the market probability will occur in the next `24h`.

**Input:**
- the current market state;
- the recent `yes_probability` trajectory;
- volatility, imbalance, and activity features;
- when available: coherence, manipulation, and external-shock proxies.

**Target:**
- a binary indicator that the future absolute move exceeds a chosen threshold.

**Why this task matters:**
- repricing can be interpreted as an observable posterior revision of the market;
- this is no longer a “trading signal,” but a task of detecting an incoming information update.

In [1]:
# If needed in a fresh notebook environment:
# %pip install -r ../requirements.txt

from __future__ import annotations

import ast
import re
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from sklearn.decomposition import TruncatedSVD
from sklearn.ensemble import HistGradientBoostingClassifier, HistGradientBoostingRegressor
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, brier_score_loss, log_loss, roc_auc_score
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

REPO_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "benchmarks" else Path.cwd().resolve()
sys.path.insert(0, str(REPO_ROOT))
sys.path.insert(0, str(REPO_ROOT / "benchmarks"))

from benchmark_utils import (
    DEFAULT_DB_PATH,
    add_time_features,
    build_multi_horizon_terminal_dataset,
    build_repricing_dataset,
    connect,
    load_eligible_markets,
    load_probabilities_for_markets,
    rolling_time_splits,
)
from covariate_utils import (
    add_lagged_covariate_features,
    asof_join_covariates,
    load_external_covariates,
    pivot_covariates_to_wide,
)

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 160)

DB_PATH = DEFAULT_DB_PATH
DOMAINS = ("geopolitics", "finance_economy")
EXTERNAL_COVARIATES_PATH = REPO_ROOT / "data" / "external_covariates"
TAG_RE = re.compile(r"[A-Za-z][A-Za-z0-9_+-]+")
EVENT_PATTERNS = {
    "election": r"election|president|prime minister|mayor|governor|parliament",
    "military": r"strike|war|missile|ceasefire|attack|troops|nuclear",
    "policy": r"tariff|fed|rate|ban|approval|regulation|sanction|etf",
    "corporate": r"earnings|revenue|ipo|acquisition|merger|bankruptcy",
    "crypto": r"bitcoin|btc|ethereum|eth|solana|crypto|token|airdrop",
}
CANDIDATE_PATTERNS = (
    re.compile(r"^will .+ be elected the next president of (.+)$"),
    re.compile(r"^will .+ win the (.+) election$"),
    re.compile(r"^which party will win (.+)$"),
)


def parse_listish(value: object) -> list[str]:
    if value is None or (isinstance(value, float) and np.isnan(value)):
        return []
    if isinstance(value, list):
        return [str(x) for x in value]
    text = str(value).strip()
    if not text:
        return []
    try:
        parsed = ast.literal_eval(text)
        if isinstance(parsed, list):
            return [str(x) for x in parsed]
    except Exception:
        pass
    return [chunk.strip() for chunk in text.split(",") if chunk.strip()]


def slug_family(slug: object) -> str:
    text = str(slug or "").lower()
    text = re.sub(r"-\d+(?:-\d+)+$", "", text)
    text = re.split(r"-(?:by|before|after|on|during|in)-", text, maxsplit=1)[0]
    text = re.sub(r"-(?:jan|january|feb|february|mar|march|apr|april|may|jun|june|jul|july|aug|august|sep|sept|september|oct|october|nov|november|dec|december).*", "", text)
    text = re.sub(r"-+$", "", text)
    return text or "unknown_family"


def candidate_family(question: object) -> str:
    q = re.sub(r"\s+", " ", str(question or "").lower()).strip(" ?")
    for pattern in CANDIDATE_PATTERNS:
        match = pattern.match(q)
        if match:
            return match.group(1).strip()
    return ""


def build_market_text(markets_df: pd.DataFrame) -> pd.DataFrame:
    work = markets_df.copy()
    work["tag_list"] = work["tag_labels"].apply(parse_listish)
    matched_domains = work["matched_domains"] if "matched_domains" in work.columns else pd.Series("", index=work.index)
    work["matched_domain_list"] = matched_domains.apply(parse_listish)
    work["question"] = work["question"].fillna("")
    work["description"] = work["description"].fillna("")
    work["resolution_source"] = work["resolution_source"].fillna("")
    work["market_text"] = (
        work["question"]
        + " [SEP] "
        + work["description"]
        + " [SEP] tags: "
        + work["tag_list"].apply(lambda values: " ".join(values))
        + " [SEP] source: "
        + work["resolution_source"]
    )
    work["family_id"] = work["market_slug"].apply(slug_family)
    work["candidate_family_id"] = work["question"].apply(candidate_family)
    work["tag_count"] = work["tag_list"].apply(len)
    work["matched_domain_count"] = work["matched_domain_list"].apply(len)
    work["question_char_len"] = work["question"].str.len()
    work["description_char_len"] = work["description"].str.len()
    work["has_resolution_source"] = work["resolution_source"].ne("").astype(int)
    for keyword, pattern in EVENT_PATTERNS.items():
        work[f"kw_{keyword}"] = work["market_text"].str.contains(pattern, case=False, regex=True).astype(int)

    vectorizer = TfidfVectorizer(stop_words="english", ngram_range=(1, 2), min_df=2, max_features=2500)
    tfidf = vectorizer.fit_transform(work["market_text"])
    if tfidf.shape[1] >= 2 and len(work) >= 3:
        n_components = int(min(16, len(work) - 1, tfidf.shape[1] - 1))
        svd = TruncatedSVD(n_components=n_components, random_state=42)
        embeddings = svd.fit_transform(tfidf)
        for idx in range(embeddings.shape[1]):
            work[f"text_svd_{idx:02d}"] = embeddings[:, idx]
        similarity = cosine_similarity(embeddings)
        np.fill_diagonal(similarity, -1.0)
        work["duplicate_neighbor_count"] = (similarity >= 0.82).sum(axis=1)
        work["max_text_similarity"] = np.where(len(work) > 1, similarity.max(axis=1), 0.0)
    else:
        work["duplicate_neighbor_count"] = 0
        work["max_text_similarity"] = 0.0

    family_stats = (
        work.groupby("family_id", dropna=False)
        .agg(
            family_market_count=("market_id", "size"),
            family_volume_sum=("volume_num", "sum"),
            family_end_date_span_days=("end_date", lambda s: (s.max() - s.min()).total_seconds() / 86400.0 if len(s) > 1 else 0.0),
        )
        .reset_index()
    )
    candidate_stats = (
        work.loc[work["candidate_family_id"].ne("")]
        .groupby("candidate_family_id", dropna=False)
        .agg(candidate_market_count=("market_id", "size"), candidate_volume_sum=("volume_num", "sum"))
        .reset_index()
    )
    work = work.merge(family_stats, on="family_id", how="left")
    work = work.merge(candidate_stats, on="candidate_family_id", how="left")
    work["candidate_market_count"] = work["candidate_market_count"].fillna(0)
    work["candidate_volume_sum"] = work["candidate_volume_sum"].fillna(0.0)
    return work


def load_multi_domain_markets(conn, *, domains, max_markets_per_domain, min_probability_rows):
    frames = []
    for domain in domains:
        frame = load_eligible_markets(
            conn,
            domain=domain,
            max_markets=max_markets_per_domain,
            min_probability_rows=min_probability_rows,
        )
        frames.append(frame)
    markets_df = pd.concat(frames, ignore_index=True)
    markets_df = markets_df.sort_values(["primary_domain", "volume_num", "created_at"], ascending=[True, False, False], kind="stable")
    markets_df = markets_df.drop_duplicates(subset=["market_id"], keep="first").reset_index(drop=True)
    return build_market_text(markets_df)


def load_optional_covariates(path: Path):
    if not path.exists():
        return None
    covariates = load_external_covariates(path)
    if covariates.empty:
        return None
    value_col = "close" if "close" in covariates.columns else "value"
    wide = pivot_covariates_to_wide(covariates, value_col=value_col)
    features = add_lagged_covariate_features(wide, lags=(1, 12, 288), pct_change=True)
    return features


def join_covariates(base_df: pd.DataFrame, covariate_df: pd.DataFrame | None, *, time_col: str) -> pd.DataFrame:
    if covariate_df is None or covariate_df.empty:
        return base_df
    return asof_join_covariates(base_df, covariate_df, base_time_col=time_col, max_age="7D")


def add_manipulation_proxies(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    eps = 1e-6
    if "lookback_24h_trade_count_sum" in out.columns:
        trade_count = out["lookback_24h_trade_count_sum"].fillna(0.0)
        total_size = out["lookback_24h_total_size_sum"].fillna(0.0)
        volatility = out["lookback_24h_volatility"].fillna(0.0)
        abs_move = out["lookback_24h_abs_move_mean"].fillna(0.0)
        max_move = out["lookback_24h_abs_move_max"].fillna(0.0)
        directional = out["lookback_24h_yes_probability_change"].fillna(0.0)
        observed_share = out["lookback_24h_observed_trade_share"].fillna(0.0)
        staleness = out.get("snapshot_staleness_hours", pd.Series(0.0, index=out.index)).fillna(0.0)
    else:
        trade_count = out["trade_count_sum"].fillna(0.0)
        total_size = out["total_size_sum"].fillna(0.0)
        volatility = out["recent_volatility"].fillna(0.0)
        abs_move = out["recent_abs_move_mean"].fillna(0.0)
        max_move = out["recent_abs_move_max"].fillna(0.0)
        directional = out["recent_directional_move"].fillna(0.0)
        observed_share = out["observed_trade_share"].fillna(0.0)
        staleness = pd.Series(0.0, index=out.index)

    confidence = out["confidence_margin"].fillna(0.0)
    out["avg_trade_size_proxy"] = total_size / (trade_count + eps)
    out["move_per_trade_proxy"] = directional.abs() / (trade_count + eps)
    out["move_per_dollar_proxy"] = directional.abs() / (total_size + eps)
    out["burstiness_proxy"] = max_move / (abs_move + eps)
    out["wash_proxy"] = observed_share * trade_count / (directional.abs() + volatility + 1e-4)
    out["conviction_without_flow_proxy"] = confidence / (observed_share + 0.05)
    out["stale_conviction_proxy"] = confidence * (1.0 + staleness)
    out["volatility_gap_proxy"] = volatility / (observed_share + 0.05)
    return out


def latest_probability_before(panel: pd.DataFrame, cutoff: pd.Timestamp) -> float:
    history = panel.loc[panel["timestamp_utc"] <= cutoff]
    if history.empty:
        return float("nan")
    return float(history["yes_probability"].iloc[-1])


def attach_family_snapshot_features(dataset: pd.DataFrame, probabilities_df: pd.DataFrame, markets_df: pd.DataFrame, *, time_col: str, prob_col: str) -> pd.DataFrame:
    panels = {market_id: frame.reset_index(drop=True) for market_id, frame in probabilities_df.groupby("market_id", sort=False)}
    family_members = markets_df.groupby("family_id", dropna=False)["market_id"].agg(list).to_dict()
    candidate_members = (
        markets_df.loc[markets_df["candidate_family_id"].ne("")]
        .groupby("candidate_family_id", dropna=False)["market_id"].agg(list).to_dict()
    )
    family_map = markets_df.set_index("market_id")["family_id"].to_dict()
    candidate_map = markets_df.set_index("market_id")["candidate_family_id"].to_dict()
    end_date_map = markets_df.set_index("market_id")["end_date"].to_dict()

    rows = []
    for row in dataset[["market_id", time_col, prob_col]].itertuples(index=False):
        market_id = str(row.market_id)
        cutoff = pd.Timestamp(getattr(row, time_col))
        current_prob = float(getattr(row, prob_col))
        family_id = family_map.get(market_id, "unknown_family")
        sibling_probs = []
        earlier_probs = []
        later_probs = []
        for sibling_id in family_members.get(family_id, []):
            if sibling_id == market_id:
                continue
            sibling_prob = latest_probability_before(panels[sibling_id], cutoff) if sibling_id in panels else float("nan")
            if np.isnan(sibling_prob):
                continue
            sibling_probs.append(sibling_prob)
            if end_date_map.get(sibling_id) < end_date_map.get(market_id):
                earlier_probs.append(sibling_prob)
            elif end_date_map.get(sibling_id) > end_date_map.get(market_id):
                later_probs.append(sibling_prob)

        candidate_family_id = candidate_map.get(market_id, "")
        candidate_probs = []
        if candidate_family_id:
            for sibling_id in candidate_members.get(candidate_family_id, []):
                sibling_prob = latest_probability_before(panels[sibling_id], cutoff) if sibling_id in panels else float("nan")
                if not np.isnan(sibling_prob):
                    candidate_probs.append(sibling_prob)

        later_violation = max(0.0, current_prob - min(later_probs)) if later_probs else 0.0
        earlier_violation = max(0.0, max(earlier_probs) - current_prob) if earlier_probs else 0.0
        candidate_sum = float(np.sum(candidate_probs)) if candidate_probs else current_prob
        rows.append(
            {
                "market_id": market_id,
                time_col: cutoff,
                "family_snapshot_size": len(sibling_probs) + 1,
                "family_prob_mean": float(np.mean(sibling_probs)) if sibling_probs else current_prob,
                "family_prob_std": float(np.std(sibling_probs)) if sibling_probs else 0.0,
                "family_prob_gap": current_prob - (float(np.mean(sibling_probs)) if sibling_probs else current_prob),
                "family_coherence_gap": later_violation + earlier_violation,
                "family_future_monotone_violation": later_violation,
                "family_past_monotone_violation": earlier_violation,
                "candidate_prob_sum_gap": abs(candidate_sum - 1.0) if candidate_probs else 0.0,
            }
        )

    features = pd.DataFrame(rows)
    return dataset.merge(features, on=["market_id", time_col], how="left")


def add_domain_dummies(df: pd.DataFrame) -> pd.DataFrame:
    if "primary_domain" not in df.columns:
        return df
    dummies = pd.get_dummies(df["primary_domain"], prefix="domain", dtype=float)
    return pd.concat([df, dummies], axis=1)


def safe_auc(y_true, pred):
    if len(np.unique(y_true)) < 2:
        return np.nan
    return roc_auc_score(y_true, pred)


def clipped(values, eps=1e-6):
    return np.clip(np.asarray(values, dtype=float), eps, 1.0 - eps)


def summarize_by_group(df: pd.DataFrame, *, group_cols, metric_cols):
    return (
        df.groupby(group_cols, dropna=False)[metric_cols]
        .agg(["mean", "std", "min", "max"])
        .reset_index()
    )


## Dataset Construction

Here the dataset is constructed as a collection of temporal snapshots for markets.

What it contains:
- for each market, we take rolling points in time;
- over a lookback window we compute recent volatility and related features;
- over a future window we determine whether a large price revision occurred.

Main filters:
- sufficient history length before and after the snapshot;
- only markets where the future window is still fully available;
- temporal subsampling so adjacent examples are not almost identical.

**Unit of evaluation:**
- one row = one market at one point in time, where we need to decide whether a large move will happen in the future window.

In [2]:
MAX_MARKETS_PER_DOMAIN = 90
FUTURE_HORIZON_HOURS = 24
LOOKBACK_HOURS = 24
SAMPLE_EVERY_HOURS = 12
MOVE_THRESHOLD = 0.15
MIN_PROBABILITY_ROWS = 14 * 24 * 12

conn = connect(DB_PATH)
markets_df = load_multi_domain_markets(
    conn,
    domains=DOMAINS,
    max_markets_per_domain=MAX_MARKETS_PER_DOMAIN,
    min_probability_rows=MIN_PROBABILITY_ROWS,
)
probabilities_df = load_probabilities_for_markets(conn, markets_df["market_id"].tolist())
repricing_df = build_repricing_dataset(
    markets_df,
    probabilities_df,
    future_horizon_hours=FUTURE_HORIZON_HOURS,
    lookback_hours=LOOKBACK_HOURS,
    sample_every_hours=SAMPLE_EVERY_HOURS,
    move_threshold=MOVE_THRESHOLD,
)
market_feature_cols = [
    "primary_domain",
    "market_text",
    "family_id",
    "candidate_family_id",
    "tag_count",
    "matched_domain_count",
    "question_char_len",
    "description_char_len",
    "has_resolution_source",
    "duplicate_neighbor_count",
    "max_text_similarity",
    "family_market_count",
    "family_volume_sum",
    "family_end_date_span_days",
    "candidate_market_count",
    "candidate_volume_sum",
] + [
    col for col in markets_df.columns if col.startswith("kw_") or col.startswith("text_svd_")
]
repricing_df = repricing_df.merge(markets_df[["market_id", *market_feature_cols]], on="market_id", how="left")
repricing_df = attach_family_snapshot_features(
    repricing_df,
    probabilities_df,
    markets_df,
    time_col="timestamp_utc",
    prob_col="current_yes_probability",
)
covariates_df = load_optional_covariates(EXTERNAL_COVARIATES_PATH)
repricing_df = join_covariates(repricing_df, covariates_df, time_col="timestamp_utc")
repricing_df = add_manipulation_proxies(repricing_df)
repricing_df = add_domain_dummies(repricing_df)
repricing_df["repricing_pressure_proxy"] = (
    repricing_df["family_coherence_gap"].fillna(0.0)
    + 0.5 * repricing_df["candidate_prob_sum_gap"].fillna(0.0)
    + 0.25 * repricing_df["wash_proxy"].fillna(0.0)
)

print(f"markets: {len(markets_df):,}")
print(f"repricing rows: {len(repricing_df):,}")
print("event rate:", round(float(repricing_df["target"].mean()), 4))
display(
    repricing_df.groupby("primary_domain", dropna=False)
    .agg(rows=("target", "size"), event_rate=("target", "mean"), mean_pressure=("repricing_pressure_proxy", "mean"))
    .reset_index()
    .sort_values("rows", ascending=False)
)
display(repricing_df.head())


markets: 180
repricing rows: 29,798
event rate: 0.0184


## Evaluation Protocol and Baselines

**Split:**
- only out-of-time evaluation;
- train/test are separated by the real snapshot timestamp.

**Metrics:**
- `average_precision` as the main metric for the rare event;
- `roc_auc` as a general discrimination metric;
- `log_loss` as probabilistic quality.

**Baselines:**
- a volatility-only heuristic;
- a market-only tree baseline;
- a richer multimodal / graph-aware model;
- when available, a separate shock-driven baseline.

**Leakage rule:**
- the future move is defined only over the window after the current timestamp;
- no feature may include anything from that future window.

In [3]:
feature_family_map = {
    "text": [col for col in repricing_df.columns if col.startswith("text_svd_") or col.startswith("kw_") or col in {"tag_count", "matched_domain_count", "question_char_len", "description_char_len", "has_resolution_source", "duplicate_neighbor_count", "max_text_similarity"}],
    "graph": [col for col in repricing_df.columns if col.startswith("family_") or col.startswith("candidate_") or col == "repricing_pressure_proxy"],
    "external": [col for col in repricing_df.columns if col.startswith("btc_usd") or col.startswith("eth_usd")],
    "manipulation": [col for col in repricing_df.columns if col.endswith("_proxy")],
    "domain": [col for col in repricing_df.columns if col.startswith("domain_")],
}
feature_family_map["graph"] = [
    col for col in feature_family_map["graph"]
    if pd.api.types.is_numeric_dtype(repricing_df[col])
]
base_excluded = {
    "market_id", "timestamp_utc", "end_date", "target", "future_move", "market_text", "family_id", "candidate_family_id",
    "resolution_source", "tag_labels", "matched_domains",
}
microstructure_cols = [
    col for col in repricing_df.columns
    if pd.api.types.is_numeric_dtype(repricing_df[col])
    and col not in base_excluded
    and col not in set().union(*feature_family_map.values())
]
full_feature_cols = sorted(set(microstructure_cols).union(*feature_family_map.values()))

micro_model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("clf", HistGradientBoostingClassifier(max_depth=4, learning_rate=0.05, max_iter=300, random_state=42)),
])
full_model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("clf", HistGradientBoostingClassifier(max_depth=5, learning_rate=0.04, max_iter=350, random_state=42)),
])
shock_model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(max_iter=2500, class_weight="balanced")),
])
shock_cols = sorted(set(feature_family_map["external"]).union(feature_family_map["graph"], feature_family_map["domain"]))

rows = []
for train_df, test_df, meta in rolling_time_splits(repricing_df, time_col="timestamp_utc", n_splits=4, min_train_fraction=0.5):
    y_test = test_df["target"].to_numpy(dtype=float)
    baseline_pred = clipped(np.clip(2.0 * test_df["recent_abs_move_mean"].to_numpy(dtype=float) + 2.0 * test_df["recent_volatility"].to_numpy(dtype=float), 0.0, 1.0))

    micro_model.fit(train_df[microstructure_cols], train_df["target"])
    micro_pred = clipped(micro_model.predict_proba(test_df[microstructure_cols])[:, 1])

    full_model.fit(train_df[full_feature_cols], train_df["target"])
    full_pred = clipped(full_model.predict_proba(test_df[full_feature_cols])[:, 1])

    shock_model.fit(train_df[shock_cols], train_df["target"])
    shock_pred = clipped(shock_model.predict_proba(test_df[shock_cols])[:, 1])[:,]

    pred_map = {
        "volatility_baseline": baseline_pred,
        "microstructure_hgb": micro_pred,
        "graph_external_logistic": shock_pred,
        "multimodal_hgb": full_pred,
    }
    for model_name, pred in pred_map.items():
        rows.append({
            "fold": meta["fold"],
            "model": model_name,
            "scope": "overall",
            "roc_auc": float(safe_auc(y_test, pred)),
            "average_precision": float(average_precision_score(y_test, pred)),
            "log_loss": float(log_loss(y_test, pred, labels=[0, 1])),
        })
        fold_view = test_df.assign(_pred=pred)
        for primary_domain, frame in fold_view.groupby("primary_domain", dropna=False):
            if len(frame) < 20:
                continue
            rows.append({
                "fold": meta["fold"],
                "model": model_name,
                "scope": primary_domain,
                "roc_auc": float(safe_auc(frame["target"], frame["_pred"])),
                "average_precision": float(average_precision_score(frame["target"], frame["_pred"])),
                "log_loss": float(log_loss(frame["target"], frame["_pred"], labels=[0, 1])),
            })

repricing_metrics = pd.DataFrame(rows)
display(
    repricing_metrics.loc[repricing_metrics["scope"].eq("overall")]
    .groupby("model")[["average_precision", "roc_auc", "log_loss"]]
    .mean()
    .sort_values("average_precision", ascending=False)
    .reset_index()
)
display(
    repricing_metrics.loc[repricing_metrics["scope"].ne("overall")]
    .groupby(["scope", "model"])[["average_precision", "roc_auc", "log_loss"]]
    .mean()
    .reset_index()
    .sort_values(["scope", "average_precision"], ascending=[True, False])
    .head(24)
)


## Results

This block answers the question: **can we recognize market states in advance that are about to be revised sharply?**

How to read the results:
- first compare the overall baselines;
- then inspect the domain breakdown;
- then check the descriptive plots: does repricing rate rise in high-pressure / high-shock states?

In [4]:
repricing_df["pressure_bucket"] = pd.qcut(repricing_df["repricing_pressure_proxy"], q=5, duplicates="drop")
repricing_df["coherence_bucket"] = pd.qcut(repricing_df["family_coherence_gap"].fillna(0.0), q=5, duplicates="drop")

display(
    repricing_df.groupby("pressure_bucket", dropna=False)
    .agg(event_rate=("target", "mean"), mean_future_move=("future_move", lambda s: np.mean(np.abs(s))), rows=("target", "size"))
    .reset_index()
)

display(
    repricing_df.groupby("coherence_bucket", dropna=False)
    .agg(event_rate=("target", "mean"), mean_future_move=("future_move", lambda s: np.mean(np.abs(s))), rows=("target", "size"))
    .reset_index()
)

plot_df = repricing_metrics.loc[repricing_metrics["scope"].eq("overall")].groupby("model")[["average_precision", "roc_auc"]].mean().reset_index()
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.barplot(data=plot_df, x="model", y="average_precision", ax=axes[0], palette="mako")
axes[0].set_title("Overall average precision")
axes[0].tick_params(axis="x", rotation=25)
sns.barplot(data=plot_df, x="model", y="roc_auc", ax=axes[1], palette="rocket")
axes[1].set_title("Overall ROC-AUC")
axes[1].tick_params(axis="x", rotation=25)
plt.tight_layout()
plt.show()


/var/folders/zv/mjsbg2dx4q79c1gbwtlhbk180000gp/T/ipykernel_12252/1405294304.py:18: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(data=plot_df, x="model", y="average_precision", ax=axes[0], palette="mako")
/var/folders/zv/mjsbg2dx4q79c1gbwtlhbk180000gp/T/ipykernel_12252/1405294304.py:21: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(data=plot_df, x="model", y="roc_auc", ax=axes[1], palette="rocket")


## Interpretation

A strong result here looks like this:
- the model captures future belief revisions better than recent volatility alone;
- repricing is not random, but concentrates in states of external shock, coherence pressure, or low trust;
- the same signal works in more than one narrow domain.

If the strongest baseline remains very simple, that does not make the benchmark weak.
It means the strongest scientific story should shift toward:
- trust estimation,
- multimodal gating,
- or structured coherence modeling,
rather than simply more complex repricing models.